# Neural Hangman — submission notebook

A bidirectional character Transformer trained from scratch on `train.txt`, supervised by an exact
dictionary-posterior oracle, corrected for covariate shift with DAgger self-play, and combined
into a calibrated ensemble.

This notebook **plays** the test set. The complete training pipeline lives in
`hangman_end_to_end.ipynb` and in `src/hangman/`; set `TRAIN = True` below to run it here instead
of loading weights.

**Design in one table** — every number measured on a 22,530-word dictionary held out of training
and disjoint from it, exactly as `test.txt` is:

| | Win rate |
|---|---:|
| Static letter frequency (what `sample_submission.csv` does) | 13.99% |
| Interpolated character n-gram, noisy-OR pooled | 60.66% |
| Exact Bayes dictionary posterior + backoff (the *teacher* that labelled training) | 61.13% |
| **This model** | see the final cell |

The two measurements that determined the design:

* `train.txt` and `test.txt` share **zero** words, so memorisation is worthless.
* **45%** of the game states a real policy visits have **no** consistent word in the training
  dictionary — dictionary filtering is silent on nearly half of them, which is why a generalising
  neural model is required rather than merely convenient.

## 1. Configuration

In [ ]:
import os, sys, json, time
from pathlib import Path

# --------------------------------------------------------------------------
# TRAIN = False -> load the trained weights and play the test set (minutes).
# TRAIN = True  -> clone the repo layout and run the full pipeline from
#                  train.txt.  Everything needed for that is in src/hangman/.
# --------------------------------------------------------------------------
TRAIN = False

ON_KAGGLE = Path("/kaggle").exists()
INPUT_ROOT = Path("/kaggle/input") if ON_KAGGLE else Path("..")

# Paths are *discovered*, not hard-coded.  The competition data mount and the
# slug of an attached dataset both vary (the slug follows whatever title you
# gave the dataset), and a wrong guess means the notebook produces no output
# file - which Kaggle reports only as "this version has no output files".
# Search at *any* depth.  Kaggle nests attached datasets under
# /kaggle/input/datasets/<owner>/<name>/versions/N/, which is five levels down,
# while competition data sits two levels down - so fixed-depth globs find one
# and miss the other.
def find_file(name: str, root: Path) -> Path | None:
    if (root / name).exists():
        return root / name
    hits = sorted(root.rglob(name))
    return hits[0] if hits else None

def find_weights(root: Path) -> Path | None:
    hits = sorted(root.rglob("*_final.pt")) or sorted(root.rglob("*.pt"))
    return hits[0].parent if hits else None

# NOTE the names.  The vendored config.py below defines module-level TEST_TXT
# and TRAIN_TXT, and everything here shares one notebook namespace - so using
# those names would have them silently overwritten a few cells later, and the
# run would fail opening a path that does not exist on Kaggle.
TEST_FILE = find_file("test.txt", INPUT_ROOT) or (INPUT_ROOT / "Data" / "test.txt")
TRAIN_FILE = find_file("train.txt", INPUT_ROOT) or (INPUT_ROOT / "Data" / "train.txt")
WEIGHTS_DIR = find_weights(INPUT_ROOT) or (Path("..") / "artifacts" / "checkpoints")
OUT_CSV = Path("/kaggle/working/submission.csv") if ON_KAGGLE else Path("../submission.csv")

print(f"on kaggle   : {ON_KAGGLE}")
print(f"test.txt    : {TEST_FILE}   exists={TEST_FILE is not None and TEST_FILE.exists()}")
print(f"weights dir : {WEIGHTS_DIR}   exists={WEIGHTS_DIR.exists()}")
if ON_KAGGLE:
    print("\ndiscovered under /kaggle/input:")
    for path in sorted(INPUT_ROOT.rglob("*")):
        if path.is_file() and path.suffix in (".txt", ".pt", ".csv", ".json"):
            print(f"  {path}")

if TEST_FILE is None or not TEST_FILE.exists():
    raise SystemExit(
        "Could not find test.txt. Attach the competition data to this notebook "
        "(Add Input -> Competitions -> brand-buzzword-hackathon)."
    )
if not WEIGHTS_DIR.exists() or not any(WEIGHTS_DIR.glob("*.pt")):
    raise SystemExit(
        "Could not find any model checkpoint. Upload the contents of kaggle_upload/ "
        "as a Dataset and attach it (Add Input -> Datasets). Without it this "
        "notebook cannot produce submission.csv."
    )

In [ ]:
import numpy as np
import pandas as pd
import torch

print(f"torch {torch.__version__}  cuda={torch.cuda.is_available()}")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Autocast dtype is chosen at *runtime*, not baked in.  This model was trained
# under bf16 on a Blackwell part, but Kaggle serves T4 (compute 7.5) and P100
# (6.0), and bf16 needs 8.0+.  Hard-coding bfloat16 here degrades or errors on
# exactly the hardware this notebook is meant to run on.
if DEVICE == "cuda":
    AMP_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    torch.backends.cuda.matmul.allow_tf32 = True
    props = torch.cuda.get_device_properties(0)
    print(f"{props.name}  sm_{props.major}{props.minor}  autocast={AMP_DTYPE}")
else:
    AMP_DTYPE = torch.float32
    print("no GPU visible - this will be slow; enable an accelerator in Settings")

## 2. Model and game engine

Vendored inline so this notebook runs with no dependency on the repository. These are the exact
files from `src/hangman/` — the only edit is that intra-package imports are stripped, since
everything lands in one namespace here.

**Architecture.** Bidirectional character Transformer, 8 blocks, `d_model` 512, SwiGLU, RMSNorm,
RoPE plus a learned absolute position embedding, ≈28.6 M parameters. Four heads on a shared trunk:
presence (the primary output), per-position (dense masked-LM supervision), occurrence-count, and
state value.

**What distinguishes it.** The guessed-letter sets — *both* the present and the **absent** ones —
are first-class inputs, injected into every block by FiLM modulation. Published hangman
transformers encode only the board; knowing that `q`, `x`, `z` are not in the word is strong
evidence about what is, and it is free.

In [ ]:
# Notebook shim.
#   * the vendored config resolves paths relative to __file__;
#   * HOLE is the hidden-position sentinel, defined in ngram.py, which is
#     not vendored here because inference needs no n-gram tables.
import pathlib
if '__file__' not in dir():
    __file__ = str(pathlib.Path.cwd() / 'notebook' / 'notebook.py')
HOLE = 27

# ====================================================================
# src/hangman/config.py
# ====================================================================
"""Configuration objects.

Every tunable in the project lives here so that experiments are described by a
single serialisable object rather than by scattered literals.  Configs are
plain dataclasses (JSON-serialisable) and are written next to each checkpoint.
"""


import dataclasses
import json
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

REPO_ROOT = Path(__file__).resolve().parents[2]
DATA_DIR = REPO_ROOT / "Data"
ARTIFACT_DIR = REPO_ROOT / "artifacts"
CHECKPOINT_DIR = ARTIFACT_DIR / "checkpoints"
DATASET_DIR = ARTIFACT_DIR / "datasets"
LOG_DIR = ARTIFACT_DIR / "logs"
METRIC_DIR = ARTIFACT_DIR / "metrics"
REPORT_DIR = REPO_ROOT / "reports"
FIGURE_DIR = REPORT_DIR / "figures"

TRAIN_TXT = DATA_DIR / "train.txt"
TEST_TXT = DATA_DIR / "test.txt"
SAMPLE_SUBMISSION_CSV = DATA_DIR / "sample_submission.csv"

#: Six wrong guesses is a loss; five is survivable (see simulator.py).
MAX_WRONG_GUESSES = 6

# Longest word in either provided file is 29 characters.  One extra slot is
# reserved for the [CLS] token, and we round to 32 for tensor-core friendliness.
MAX_WORD_LEN = 29
SEQ_LEN = 32

# Token vocabulary for the character axis.
TOK_PAD, TOK_MASK, TOK_CLS = 26, 27, 28
VOCAB_SIZE = 29

# Global feature vector layout (see state.encode_states).
#   [0:26]   absent-letter indicator
#   [26:52]  present-letter indicator
#   [52:58]  one-hot lives remaining (0..5)
#   [58]     word length / MAX_WORD_LEN
#   [59]     number of guesses so far / 26
#   [60]     fraction of positions revealed
#   [61]     number of distinct revealed letters / 26
GLOBAL_FEATURE_DIM = 62


@dataclass
class DataConfig:
    """Dictionary loading and the disjoint fit/validation split."""

    train_path: Path = TRAIN_TXT
    test_path: Path = TEST_TXT
    #: Fraction of train.txt held out as a *disjoint validation dictionary*.
    #: This mirrors the real condition: train and test share zero words.
    val_fraction: float = 0.10
    split_seed: int = 20260903
    #: When True the teacher's candidate dictionary is the full train.txt.
    #: Used only for the final submission model, never during model selection.
    use_full_dictionary: bool = False


@dataclass
class TeacherConfig:
    """Three-tier Bayes/n-gram posterior oracle."""

    #: Highest n-gram order (inclusive).  Orders 2..max are interpolated.
    max_ngram: int = 7
    #: Weight of order-n evidence in the interpolation, w(n) = n ** ngram_power.
    ngram_power: float = 3.0
    #: Jelinek-Mercer style mixing between the exact candidate posterior and the
    #: smoothed n-gram posterior: lambda = n_candidates / (n_candidates + kappa).
    backoff_kappa: float = 300.0
    #: How many rungs of the constraint-relaxation ladder to use (1..3).
    n_exact_tiers: int = 3
    #: 'linear' mixes presence marginals arithmetically; 'loglinear' pools them
    #: geometrically, which is sharper when the sources agree.
    blend: str = "linear"
    #: Laplace smoothing added to candidate presence counts.
    presence_prior: float = 0.5
    #: Leave-one-out: remove the played word from its own candidate set so the
    #: teacher faces the same unseen-word condition as test time.
    leave_one_out: bool = True


@dataclass
class ModelConfig:
    """HangmanFormer architecture."""

    name: str = "hangmanformer_m"
    arch: str = "transformer"          # transformer | bigru | dilated_cnn
    d_model: int = 512
    n_layers: int = 8
    n_heads: int = 8
    d_ff: int = 1408                   # SwiGLU: ~2.75 x d_model
    dropout: float = 0.10
    stochastic_depth: float = 0.05
    use_rope: bool = True
    use_abs_pos: bool = True
    use_film: bool = True
    #: Auxiliary heads.
    use_pos_head: bool = True
    use_count_head: bool = True
    use_value_head: bool = True
    count_buckets: int = 4             # counts {0, 1, 2, >=3}
    #: Bi-GRU / CNN specific.
    rnn_layers: int = 3
    cnn_layers: int = 8
    cnn_kernel: int = 3


@dataclass
class LossConfig:
    """Multi-task objective weights."""

    w_set_soft: float = 1.0            # BCE against the Bayes posterior
    w_set_hard: float = 0.25           # BCE against the ground-truth 0/1 label
    w_pos: float = 0.40                # per-position KL
    w_count: float = 0.15              # count-bucket CE
    w_value: float = 0.10              # win-probability BCE
    w_llm: float = 0.0                 # KL to the Qwen teacher (set in distill)
    distill_temperature: float = 2.0
    label_smoothing: float = 0.01


@dataclass
class OptimConfig:
    lr: float = 3e-4
    weight_decay: float = 0.01
    betas: tuple[float, float] = (0.9, 0.95)
    grad_clip: float = 1.0
    warmup_steps: int = 2000
    batch_size: int = 4096
    epochs: int = 1
    #: Exponential moving average of weights; 0 disables.
    ema_decay: float = 0.999
    amp_dtype: str = "bfloat16"
    #: torch.compile fuses the elementwise work (RMSNorm scaling, FiLM
    #: modulation, SwiGLU gating, residuals) that profiling showed to be 30% of
    #: step time on this machine.  Measured 2.3x faster and 37% less memory.
    compile: bool = True


@dataclass
class CollectConfig:
    """State collection for the off-policy warm start and DAgger rounds."""

    #: Number of (word, state) samples drawn in the off-policy stage.
    offpolicy_states: int = 20_000_000
    #: On-policy states collected per DAgger round.
    onpolicy_states_per_round: int = 8_000_000
    dagger_rounds: int = 4
    #: Fraction of each training batch drawn from the off-policy pool.
    offpolicy_mix: float = 0.30
    #: Oversampling weight applied to words of length 4..9, where the measured
    #: baseline loses most games.
    short_word_boost: float = 1.6
    short_word_range: tuple[int, int] = (4, 9)
    n_workers: int = 18
    seed: int = 12345


@dataclass
class SelfDistillConfig:
    """Using a trained model as the teacher for the next generation.

    The Bayes oracle that labelled generation one scores 0.656 top-1 on the
    states that matter; the model it trained scores 0.744.  Once that is true
    the oracle is no longer the best target available - see
    :mod:`hangman.selfdistill`.
    """

    enabled: bool = False
    #: Gate on candidate count: ``lambda = n / (n + kappa)`` weight on the Bayes
    #: posterior.  0 = pure student, large = pure Bayes.  Chosen by measurement
    #: in ``tools/blend_sweep.py``, recorded in artifacts/metrics/blend_sweep.json.
    kappa_sd: float = 50.0
    #: Checkpoint of the previous generation, used as the teacher.
    teacher_ckpt: str = ""
    #: Noisy-OR blend for the teacher's own inference; the value fitted for it.
    teacher_noisy_or_weight: float = 0.4


@dataclass
class PolicyConfig:
    """Guess-selection at inference time."""

    mode: str = "auto"                 # greedy | value | exact | auto
    #: Value-guided one-ply lookahead is used when lives_left <= this.
    value_lookahead_max_lives: int = 2
    value_topk: int = 5
    #: Exact information-set search runs when the consistent candidate set is
    #: non-empty and no larger than this.
    exact_max_candidates: int = 120
    #: Logit-space ensemble/source blending, fitted on the validation dictionary.
    calibrate: bool = True


@dataclass
class LLMConfig:
    """Qwen3-4B teacher."""

    model_path: str = "/home/samarthamp/ai-models/qwen3-4b"
    #: Keep only the first N transformer layers (0 = keep all).  Chosen by the
    #: layer-truncation study in reports/FINETUNING_REPORT.md.
    keep_layers: int = 18
    bidirectional: bool = True         # LLM2Vec-style causal-mask removal
    lora_r: int = 32
    lora_alpha: int = 64
    lora_dropout: float = 0.05
    lora_targets: tuple[str, ...] = (
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    )
    load_in_4bit: bool = False         # QLoRA; bitsandbytes is unreliable here
    mntp_steps: int = 4000             # masked-character adaptation
    sft_steps: int = 12000             # fit to Bayes posteriors
    batch_size: int = 64
    lr: float = 1e-4
    max_seq_len: int = 96
    #: Cap on states scored by the teacher during distillation.
    max_distill_states: int = 3_000_000


@dataclass
class RLConfig:
    """Self-play policy refinement."""

    enabled: bool = True
    algo: str = "ppo"                  # ppo | reinforce
    steps: int = 4000
    games_per_step: int = 2048
    lr: float = 2e-5
    clip_eps: float = 0.2
    entropy_coef: float = 0.005
    value_coef: float = 0.5
    #: KL trust region back to the supervised policy, preventing collapse.
    kl_coef: float = 0.05
    #: Reward = win - efficiency_penalty * wrong_guesses, matching the
    #: leaderboard tiebreak (Score - E / 1e8).
    efficiency_penalty: float = 0.001


@dataclass
class ExperimentConfig:
    """Top-level configuration for one training run."""

    run_name: str = "hangmanformer_m_seed0"
    seed: int = 0
    device: str = "cuda"
    data: DataConfig = field(default_factory=DataConfig)
    teacher: TeacherConfig = field(default_factory=TeacherConfig)
    model: ModelConfig = field(default_factory=ModelConfig)
    loss: LossConfig = field(default_factory=LossConfig)
    optim: OptimConfig = field(default_factory=OptimConfig)
    collect: CollectConfig = field(default_factory=CollectConfig)
    policy: PolicyConfig = field(default_factory=PolicyConfig)
    selfdistill: SelfDistillConfig = field(default_factory=SelfDistillConfig)
    llm: LLMConfig = field(default_factory=LLMConfig)
    rl: RLConfig = field(default_factory=RLConfig)

    def to_dict(self) -> dict[str, Any]:
        return dataclasses.asdict(self)

    def save(self, path: str | Path) -> None:
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(json.dumps(self.to_dict(), indent=2, default=str))

    @staticmethod
    def load(path: str | Path) -> "ExperimentConfig":
        return ExperimentConfig.load_from_dict(json.loads(Path(path).read_text()))

    @staticmethod
    def load_from_dict(raw: dict[str, Any]) -> "ExperimentConfig":
        return _from_dict(ExperimentConfig, raw)


def _from_dict(cls: type, raw: dict[str, Any]) -> Any:
    """Rebuild a nested dataclass from a plain dict.

    ``from __future__ import annotations`` turns every field annotation into a
    string, so ``field.type`` cannot be tested with ``is_dataclass``.  The
    nested config types are all supplied through ``default_factory``, so we
    recover the real class by calling it.
    """
    kwargs: dict[str, Any] = {}
    for f in dataclasses.fields(cls):
        if f.name not in raw:
            continue
        value = raw[f.name]
        nested = None
        if f.default_factory is not dataclasses.MISSING:  # type: ignore[misc]
            candidate = f.default_factory()               # type: ignore[misc]
            if dataclasses.is_dataclass(candidate):
                nested = type(candidate)
        if nested is not None and isinstance(value, dict):
            kwargs[f.name] = _from_dict(nested, value)
        elif f.name.endswith("_path") or f.name in {"train_path", "test_path"}:
            kwargs[f.name] = Path(value)
        elif isinstance(value, list) and isinstance(
            f.default if f.default is not dataclasses.MISSING else None, tuple
        ):
            kwargs[f.name] = tuple(value)
        else:
            kwargs[f.name] = value
    return cls(**kwargs)


def ensure_dirs() -> None:
    """Create every artifact directory the pipeline writes to."""
    for d in (ARTIFACT_DIR, CHECKPOINT_DIR, DATASET_DIR, LOG_DIR, METRIC_DIR, FIGURE_DIR):
        d.mkdir(parents=True, exist_ok=True)

In [ ]:
# ====================================================================
# src/hangman/simulator.py
# ====================================================================
"""Batched lock-step Hangman engine implementing the competition rules exactly.

Rules encoded here (from the problem statement)
-----------------------------------------------
* A correct letter reveals **all** of its occurrences at once.
* "Any character in your string that does not actively reveal a new letter in
  the hidden word - including an incorrect letter, a repeated/duplicate letter,
  a space, a number, or an invalid symbol - counts as exactly 1 wrong guess."
  So a *repeat* of an already-guessed letter is a strike even if that letter is
  in the word.  The engine implements this; our policies additionally never
  repeat, because a repeat can only ever cost a life.
* "The simulation loop for each word terminates instantly the moment your
  guessed_letters_string triggers a 6th wrong guess or completes the word."
  Six wrong guesses is therefore a **loss**; five is survivable.

Note that the blueprint snippet in the problem statement uses
``while wrong_guesses <= 6``, which would permit a seventh guess.  That
contradicts the normative rule text quoted above, so we implement the strict
rule - the conservative choice, since a policy tuned for six survivable misses
would lose games under the strict scorer but never the reverse.

Every game in a batch advances in lock-step so that a single model forward pass
serves thousands of games.  Finished games are simply masked out.
"""


from dataclasses import dataclass
from typing import Callable, Protocol

import numpy as np
import torch


#: Marker used in the board tensor for positions past the end of a short word.
PAD_POS = -1


class GuessFn(Protocol):
    """Signature of a guess policy consumed by :func:`play_batch`."""

    def __call__(
        self,
        board: torch.Tensor,      # (B, Lmax) int64, HOLE where hidden, PAD_POS past the end
        lengths: torch.Tensor,    # (B,) int64
        present: torch.Tensor,    # (B, 26) bool
        absent: torch.Tensor,     # (B, 26) bool
        lives: torch.Tensor,      # (B,) int64, guesses remaining before loss
        active: torch.Tensor,     # (B,) bool
    ) -> torch.Tensor:            # (B,) int64 letter ids
        ...


@dataclass
class GameResult:
    """Outcome of a batch of games."""

    won: torch.Tensor            # (B,) bool
    wrong: torch.Tensor          # (B,) int64  strikes accumulated
    n_guesses: torch.Tensor      # (B,) int64  length of the guess string
    guess_seq: torch.Tensor      # (B, 32) int8, -1 padded, in order

    def win_rate(self) -> float:
        return float(self.won.to(torch.float64).mean().item())

    def total_wrong(self) -> int:
        """``E`` in the leaderboard tiebreak ``Score - E / 1e8``."""
        return int(self.wrong.sum().item())

    def guess_strings(self) -> list[str]:
        seq = self.guess_seq.detach().cpu().numpy()
        counts = self.n_guesses.detach().cpu().numpy()
        out = []
        for row, n in zip(seq, counts):
            out.append("".join(chr(97 + int(c)) for c in row[:n]))
        return out


class BatchGame:
    """Mutable state for a batch of simultaneous games."""

    def __init__(self, words: list[str], device: str | torch.device = "cuda") -> None:
        self.device = torch.device(device)
        self.words = words
        batch = len(words)
        lengths = np.array([len(w) for w in words], dtype=np.int64)
        lmax = int(lengths.max()) if batch else 1
        if lmax > MAX_WORD_LEN:
            raise ValueError(f"word longer than MAX_WORD_LEN={MAX_WORD_LEN}")

        truth = np.full((batch, lmax), PAD_POS, dtype=np.int64)
        presence = np.zeros((batch, 26), dtype=bool)
        for i, word in enumerate(words):
            ids = np.frombuffer(word.encode("ascii"), dtype=np.uint8).astype(np.int64) - 97
            truth[i, : ids.size] = ids
            presence[i, ids] = True

        self.truth = torch.from_numpy(truth).to(self.device)
        self.lengths = torch.from_numpy(lengths).to(self.device)
        self.truth_presence = torch.from_numpy(presence).to(self.device)

        self.present = torch.zeros((batch, 26), dtype=torch.bool, device=self.device)
        self.absent = torch.zeros((batch, 26), dtype=torch.bool, device=self.device)
        self.wrong = torch.zeros(batch, dtype=torch.long, device=self.device)
        self.n_guesses = torch.zeros(batch, dtype=torch.long, device=self.device)
        # 26 distinct letters plus up to 5 survivable strikes; 32 is a safe cap.
        self.guess_seq = torch.full((batch, 32), -1, dtype=torch.int8, device=self.device)
        self._rows = torch.arange(batch, device=self.device)

    # -- observations --------------------------------------------------------

    @property
    def batch_size(self) -> int:
        return int(self.truth.shape[0])

    def board(self) -> torch.Tensor:
        """``(B, Lmax)`` board: true letter where revealed, ``HOLE`` where hidden."""
        real = self.truth >= 0
        revealed = self.present.gather(1, self.truth.clamp_min(0)) & real
        board = torch.where(revealed, self.truth, torch.full_like(self.truth, HOLE))
        return torch.where(real, board, torch.full_like(self.truth, PAD_POS))

    def solved(self) -> torch.Tensor:
        """``(B,)`` bool: every letter of the word has been revealed."""
        return (self.truth_presence & ~self.present).sum(dim=1) == 0

    def lives(self) -> torch.Tensor:
        """``(B,)`` remaining strikes before the game is lost."""
        return MAX_WRONG_GUESSES - self.wrong

    def active(self) -> torch.Tensor:
        """``(B,)`` bool: game still in progress."""
        return (self.wrong < MAX_WRONG_GUESSES) & ~self.solved()

    # -- transition ----------------------------------------------------------

    def apply(self, guesses: torch.Tensor, active: torch.Tensor | None = None) -> torch.Tensor:
        """Apply one guess per game.  Returns the ``(B,)`` bool hit mask.

        A guess counts as a hit only if the letter is in the word **and** has
        not been guessed before; anything else is a strike, per the rules.
        """
        if active is None:
            active = self.active()
        rows = self._rows[active]
        if rows.numel() == 0:
            return torch.zeros(self.batch_size, dtype=torch.bool, device=self.device)

        letters = guesses[active]
        already = self.present[rows, letters] | self.absent[rows, letters]
        in_word = self.truth_presence[rows, letters]
        hit = in_word & ~already

        # Clamp the write index: 26 distinct letters plus five survivable
        # strikes cannot exceed 31, but a stochastic policy that somehow
        # repeated itself would index out of bounds, and an out-of-bounds
        # scatter on CUDA is a device-side assert that surfaces asynchronously
        # somewhere else entirely.
        slot = self.n_guesses[rows].clamp(max=self.guess_seq.shape[1] - 1)
        self.guess_seq[rows, slot] = letters.to(torch.int8)
        self.n_guesses[rows] += 1

        self.present[rows[hit], letters[hit]] = True
        # A letter not in the word is recorded absent whether or not it is a
        # repeat; a repeat of a *correct* letter needs no state change, it just
        # costs a life.
        not_in_word = ~in_word
        self.absent[rows[not_in_word], letters[not_in_word]] = True
        self.wrong[rows[~hit]] += 1

        full_hit = torch.zeros(self.batch_size, dtype=torch.bool, device=self.device)
        full_hit[rows] = hit
        return full_hit

    def result(self) -> GameResult:
        return GameResult(
            won=self.solved(),
            wrong=self.wrong.clone(),
            n_guesses=self.n_guesses.clone(),
            guess_seq=self.guess_seq.clone(),
        )


@torch.no_grad()
def play_batch(
    words: list[str],
    guess_fn: GuessFn,
    device: str | torch.device = "cuda",
    max_turns: int = 26,
    on_state: Callable[[BatchGame, torch.Tensor], None] | None = None,
) -> GameResult:
    """Play a batch of games to completion under ``guess_fn``.

    ``on_state`` is invoked before each turn with the game and the active mask,
    which is how DAgger collects the on-policy state distribution.
    """
    game = BatchGame(words, device=device)
    for _ in range(max_turns):
        active = game.active()
        if not bool(active.any()):
            break
        if on_state is not None:
            on_state(game, active)
        guesses = guess_fn(
            board=game.board(),
            lengths=game.lengths,
            present=game.present,
            absent=game.absent,
            lives=game.lives(),
            active=active,
        )
        game.apply(guesses, active)
    return game.result()


def reference_play(word: str, guess_order: str) -> tuple[bool, int, int]:
    """Naive single-game scorer, used to validate the batched engine.

    Returns ``(won, wrong, n_consumed)`` for a fixed guess string, applying the
    6-strike lockout exactly as the competition scorer describes it.
    """
    revealed: set[str] = set()
    guessed: set[str] = set()
    wrong = 0
    for i, ch in enumerate(guess_order):
        if wrong >= MAX_WRONG_GUESSES or not (set(word) - revealed):
            return (not (set(word) - revealed)), wrong, i
        if ch in word and ch not in guessed:
            revealed.add(ch)
        else:
            wrong += 1
        guessed.add(ch)
    return (not (set(word) - revealed)), wrong, len(guess_order)

In [ ]:
# ====================================================================
# src/hangman/state.py
# ====================================================================
"""Game state -> model tensors.

A hangman state is ``(board, present letters, absent letters, lives)``.  Two
observations drive the encoding:

1. **The absent set is evidence, not bookkeeping.**  Knowing that ``q``, ``x``
   and ``z`` are *not* in the word sharply reshapes the posterior over what is.
   Public hangman transformers encode only the board and at best suppress
   already-guessed letters at the output; the surveyed reference implementation
   states outright that it "doesn't account for letters guessed but absent".
   We feed both guessed sets in as first-class inputs.

2. **The guessed sets are sets.**  They carry no order, so they are encoded as
   permutation-invariant indicator vectors rather than as a sequence
   (Zaheer et al., 2017, *Deep Sets*).

Everything the model needs is derivable from ``(word, guessed_letter_mask)``,
which is what makes the training records in :mod:`hangman.collect` only eight
bytes of state each.
"""


import numpy as np
import torch



def encode_tokens(board: torch.Tensor, seq_len: int | None = None) -> tuple[torch.Tensor, torch.Tensor]:
    """Board -> ``(tokens, attention_mask)`` of shape ``(B, seq_len)``.

    Slot 0 is the ``[CLS]`` summary token; slots ``1..L`` hold the board, with
    ``TOK_MASK`` for hidden positions and ``TOK_PAD`` beyond the word.

    ``seq_len`` defaults to whatever this batch actually needs rather than the
    global maximum of 32.  Words average 9.4 characters, so padding every batch
    to 32 would trip the model's cost - which on this hardware is dominated by
    activation traffic, not arithmetic.  Combined with the length-bucketed
    sampler in :mod:`hangman.dataset`, batches are typically 10-12 wide.
    """
    batch, lmax = board.shape
    device = board.device
    if seq_len is None:
        seq_len = min(SEQ_LEN, lmax + 1)
    tokens = torch.full((batch, seq_len), TOK_PAD, dtype=torch.long, device=device)
    tokens[:, 0] = TOK_CLS

    width = min(lmax, seq_len - 1)
    chunk = board[:, :width]
    body = torch.where(
        chunk == PAD_POS,
        torch.full_like(chunk, TOK_PAD),
        torch.where(chunk == HOLE, torch.full_like(chunk, TOK_MASK), chunk),
    )
    tokens[:, 1:1 + width] = body

    attn = tokens != TOK_PAD
    attn[:, 0] = True
    return tokens, attn


def encode_globals(
    lengths: torch.Tensor,
    present: torch.Tensor,
    absent: torch.Tensor,
) -> torch.Tensor:
    """``(B, GLOBAL_FEATURE_DIM)`` float32 conditioning vector.

    Layout (see :mod:`hangman.config`)::

        [ 0:26]  absent indicator
        [26:52]  present indicator
        [52:58]  one-hot lives remaining, clamped to 0..5
        [58]     word length / MAX_WORD_LEN
        [59]     guesses made / 26
        [60]     fraction of positions revealed
        [61]     distinct revealed letters / 26
    """
    device = present.device
    batch = present.shape[0]
    absent_f = absent.to(torch.float32)
    present_f = present.to(torch.float32)

    wrong = absent_f.sum(dim=1)
    lives = (MAX_WRONG_GUESSES - wrong).clamp(0, MAX_WRONG_GUESSES - 1).to(torch.long)
    lives_onehot = torch.zeros((batch, MAX_WRONG_GUESSES), dtype=torch.float32, device=device)
    lives_onehot.scatter_(1, lives.unsqueeze(1), 1.0)

    n_guesses = present_f.sum(dim=1) + wrong
    lengths_f = lengths.to(torch.float32)

    # Fraction of *positions* revealed needs the per-letter multiplicity, which
    # the caller does not have; the distinct-letter proxy below is equivalent
    # information for the network and costs nothing to compute.
    revealed_letters = present_f.sum(dim=1)

    extras = torch.stack(
        [
            lengths_f / float(MAX_WORD_LEN),
            n_guesses / 26.0,
            revealed_letters / lengths_f.clamp_min(1.0),
            revealed_letters / 26.0,
        ],
        dim=1,
    )
    return torch.cat([absent_f, present_f, lives_onehot, extras], dim=1)


def encode_batch(
    board: torch.Tensor,
    lengths: torch.Tensor,
    present: torch.Tensor,
    absent: torch.Tensor,
) -> dict[str, torch.Tensor]:
    """Full model input for a batch of states."""
    tokens, attn = encode_tokens(board)
    return {
        "tokens": tokens,
        "attn_mask": attn,
        "globals": encode_globals(lengths, present, absent),
        "guessed": present | absent,
    }


def board_from_records(
    words: torch.Tensor,
    lengths: torch.Tensor,
    guessed: torch.Tensor,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Reconstruct ``(board, present, absent)`` from compact training records.

    Training records store only the word and a 26-bit guessed-letter mask; the
    board, the present/absent split and the remaining lives all follow
    deterministically, which keeps a 20M-state dataset near 1.3 GB.

    Parameters
    ----------
    words : (B, Lmax) int64 letter ids, ``PAD_POS`` beyond the word.
    lengths : (B,) int64.
    guessed : (B, 26) bool.
    """
    real = words >= 0
    word_presence = torch.zeros_like(guessed)
    word_presence.scatter_(1, words.clamp_min(0), True)
    # Words shorter than Lmax scatter a spurious True at column 0; undo it.
    has_a = (words == 0).any(dim=1)
    word_presence[:, 0] &= has_a

    present = guessed & word_presence
    absent = guessed & ~word_presence

    revealed = present.gather(1, words.clamp_min(0)) & real
    board = torch.where(revealed, words, torch.full_like(words, HOLE))
    board = torch.where(real, board, torch.full_like(words, PAD_POS))
    return board, present, absent


def pack_words(words: list[str], device: torch.device | str = "cpu") -> tuple[torch.Tensor, torch.Tensor]:
    """``(word_ids, lengths)`` tensors for a list of words."""
    lengths = np.array([len(w) for w in words], dtype=np.int64)
    lmax = int(lengths.max()) if len(words) else 1
    out = np.full((len(words), lmax), PAD_POS, dtype=np.int64)
    for i, w in enumerate(words):
        out[i, : len(w)] = np.frombuffer(w.encode("ascii"), dtype=np.uint8).astype(np.int64) - 97
    return torch.from_numpy(out).to(device), torch.from_numpy(lengths).to(device)


def guessed_mask_to_bits(mask: torch.Tensor) -> torch.Tensor:
    """``(B, 26)`` bool -> ``(B,)`` uint32 bitmask, for compact storage."""
    weights = (1 << torch.arange(26, device=mask.device, dtype=torch.long))
    return (mask.to(torch.long) * weights).sum(dim=1).to(torch.int64)


def bits_to_guessed_mask(bits: torch.Tensor) -> torch.Tensor:
    """Inverse of :func:`guessed_mask_to_bits`."""
    shifts = torch.arange(26, device=bits.device, dtype=torch.long)
    return ((bits.unsqueeze(1) >> shifts) & 1).to(torch.bool)

In [ ]:
# ====================================================================
# src/hangman/models/layers.py
# ====================================================================
"""Shared building blocks: RMSNorm, SwiGLU, rotary embeddings, FiLM modulation."""


import math

import torch
import torch.nn as nn
import torch.nn.functional as F


class RMSNorm(nn.Module):
    """Root-mean-square layer norm (Zhang & Sennrich, 2019).

    Cheaper than LayerNorm and empirically as stable in pre-norm transformers.
    """

    def __init__(self, dim: int, eps: float = 1e-6) -> None:
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # ``sum(dtype=float32)`` accumulates in fp32 without materialising an
        # fp32 copy of ``x``.  On this machine (273 GB/s against ~100 TFLOPS)
        # the model is memory-bound, so avoiding that copy is worth more than
        # the arithmetic it saves.
        var = x.pow(2).sum(-1, keepdim=True, dtype=torch.float32) / x.shape[-1]
        return x * torch.rsqrt(var + self.eps).to(x.dtype) * self.weight


class SwiGLU(nn.Module):
    """Gated feed-forward block (Shazeer, 2020, *GLU Variants Improve Transformer*)."""

    def __init__(self, dim: int, hidden: int, dropout: float = 0.0) -> None:
        super().__init__()
        self.gate = nn.Linear(dim, hidden, bias=False)
        self.up = nn.Linear(dim, hidden, bias=False)
        self.down = nn.Linear(hidden, dim, bias=False)
        self.drop = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.drop(self.down(F.silu(self.gate(x)) * self.up(x)))


class RotaryEmbedding(nn.Module):
    """Rotary position embedding (Su et al., 2021).

    Applied on the character axis so that the model sees *relative* offsets
    between letters - the signal that digraph and affix patterns live in -
    while the learned absolute embedding separately marks word-initial and
    word-final positions.
    """

    def __init__(self, head_dim: int, max_len: int, base: float = 10_000.0) -> None:
        super().__init__()
        if head_dim % 2 != 0:
            raise ValueError("rotary head_dim must be even")
        inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
        pos = torch.arange(max_len).float()
        freqs = torch.outer(pos, inv_freq)
        self.register_buffer("cos", freqs.cos()[None, None, :, :], persistent=False)
        self.register_buffer("sin", freqs.sin()[None, None, :, :], persistent=False)

    @staticmethod
    def _rotate(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
        x1, x2 = x[..., 0::2], x[..., 1::2]
        out = torch.empty_like(x)
        out[..., 0::2] = x1 * cos - x2 * sin
        out[..., 1::2] = x1 * sin + x2 * cos
        return out

    def forward(self, q: torch.Tensor, k: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        seq = q.shape[-2]
        cos = self.cos[:, :, :seq, :].to(q.dtype)
        sin = self.sin[:, :, :seq, :].to(q.dtype)
        return self._rotate(q, cos, sin), self._rotate(k, cos, sin)


class FiLMGenerator(nn.Module):
    """Produces per-layer scale/shift pairs from the global conditioning vector.

    Feature-wise Linear Modulation (Perez et al., 2018).  The guessed-letter
    sets and remaining lives are *global* constraints on the whole board, so
    injecting them only at the ``[CLS]`` token would force the network to route
    them through attention.  FiLM instead lets them modulate every block
    directly.  Initialised to zero so the network starts as an unconditioned
    transformer and learns to use the conditioning.
    """

    def __init__(self, cond_dim: int, d_model: int, n_layers: int, bottleneck: int = 128) -> None:
        super().__init__()
        self.n_layers = n_layers
        self.d_model = d_model
        self.trunk = nn.Sequential(
            nn.Linear(cond_dim, bottleneck),
            nn.SiLU(),
            nn.Linear(bottleneck, bottleneck),
            nn.SiLU(),
        )
        self.heads = nn.ModuleList(
            [nn.Linear(bottleneck, 4 * d_model) for _ in range(n_layers)]
        )
        for head in self.heads:
            nn.init.zeros_(head.weight)
            nn.init.zeros_(head.bias)

    def forward(self, cond: torch.Tensor) -> list[tuple[torch.Tensor, ...]]:
        h = self.trunk(cond)
        out = []
        for head in self.heads:
            gamma_a, beta_a, gamma_f, beta_f = head(h).chunk(4, dim=-1)
            out.append((gamma_a.unsqueeze(1), beta_a.unsqueeze(1),
                        gamma_f.unsqueeze(1), beta_f.unsqueeze(1)))
        return out


class EncoderBlock(nn.Module):
    """Pre-norm bidirectional transformer block with adaLN-style FiLM."""

    def __init__(
        self,
        d_model: int,
        n_heads: int,
        d_ff: int,
        dropout: float,
        drop_path: float,
        rope: RotaryEmbedding | None,
    ) -> None:
        super().__init__()
        if d_model % n_heads:
            raise ValueError("d_model must be divisible by n_heads")
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.rope = rope
        self.drop_path = drop_path

        self.norm_attn = RMSNorm(d_model)
        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.proj = nn.Linear(d_model, d_model, bias=False)
        self.attn_drop = dropout

        self.norm_ffn = RMSNorm(d_model)
        self.ffn = SwiGLU(d_model, d_ff, dropout)
        self.resid_drop = nn.Dropout(dropout)

    def _stochastic_depth(self, branch: torch.Tensor) -> torch.Tensor:
        if not self.training or self.drop_path <= 0.0:
            return branch
        keep = 1.0 - self.drop_path
        mask = torch.rand(branch.shape[0], 1, 1, device=branch.device) < keep
        return branch * mask / keep

    def forward(
        self,
        x: torch.Tensor,
        pad_mask: torch.Tensor,
        film: tuple[torch.Tensor, ...] | None,
    ) -> torch.Tensor:
        gamma_a, beta_a, gamma_f, beta_f = (
            film if film is not None else (0.0, 0.0, 0.0, 0.0)
        )

        h = self.norm_attn(x)
        h = h * (1.0 + gamma_a) + beta_a
        batch, seq, _ = h.shape
        qkv = self.qkv(h).view(batch, seq, 3, self.n_heads, self.head_dim)
        q, k, v = qkv.permute(2, 0, 3, 1, 4)
        if self.rope is not None:
            q, k = self.rope(q, k)
        attn_bias = pad_mask[:, None, None, :]
        attended = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask=attn_bias,
            dropout_p=self.attn_drop if self.training else 0.0,
        )
        attended = attended.transpose(1, 2).reshape(batch, seq, -1)
        x = x + self._stochastic_depth(self.resid_drop(self.proj(attended)))

        h = self.norm_ffn(x)
        h = h * (1.0 + gamma_f) + beta_f
        return x + self._stochastic_depth(self.ffn(h))


def noisy_or_presence(pos_logits: torch.Tensor, hidden_mask: torch.Tensor) -> torch.Tensor:
    """Pool per-position letter distributions into presence probabilities.

    ``P(letter present) = 1 - prod_over_hidden_positions (1 - p_i(letter))``,
    evaluated in log space for stability.  This is the correct probabilistic
    aggregation; the max-pooling used by public hangman transformers
    systematically understates letters that are likely to occur more than once.

    Parameters
    ----------
    pos_logits : (B, L, 26)
    hidden_mask : (B, L) bool, True at positions still hidden.
    """
    probs = pos_logits.float().softmax(dim=-1)
    log_miss = torch.log1p(-probs.clamp(max=1.0 - 1e-6))
    log_miss = log_miss * hidden_mask.unsqueeze(-1).to(log_miss.dtype)
    return -torch.expm1(log_miss.sum(dim=1))


def count_parameters(module: nn.Module) -> int:
    return sum(p.numel() for p in module.parameters() if p.requires_grad)


def layer_table(model: nn.Module) -> list[tuple[str, str, int]]:
    """``(name, shape, count)`` for every top-level submodule, for the report."""
    rows: list[tuple[str, str, int]] = []
    for name, child in model.named_children():
        n = count_parameters(child)
        shapes = {tuple(p.shape) for p in child.parameters()}
        shape_str = ", ".join(str(s) for s in sorted(shapes)[:3])
        if len(shapes) > 3:
            shape_str += ", ..."
        rows.append((name, shape_str, n))
    return rows

In [ ]:
# ====================================================================
# src/hangman/models/hangmanformer.py
# ====================================================================
"""HangmanFormer: a bidirectional character transformer for hangman.

Design summary
--------------
* **Bidirectional, not causal.**  Reading ``_ o _ _ e r`` requires the suffix as
  much as the prefix, so the encoder attends both ways, BERT-style
  (Devlin et al., 2019) over a character alphabet.  Character granularity is
  the right level for a spelling task - the motivation behind CANINE
  (Clark et al., 2021), ByT5 (Xue et al., 2021) and Charformer (Tay et al.,
  2022).
* **Global constraints via FiLM.**  Guessed-letter sets and remaining lives
  modulate every block (Perez et al., 2018) instead of being appended to the
  sequence.
* **Four heads, one shared trunk.**  A presence head is what the game actually
  needs; a per-position head supplies dense supervision; a count head tells the
  policy how much a hit reveals; a value head estimates the win probability so
  the endgame can be played for win rate rather than for hit rate.
* **Noisy-OR pooling.**  The per-position distributions are pooled into a
  second presence estimate with ``1 - prod(1 - p)``, which is the correct
  probabilistic aggregation.  Public implementations max-pool here, which
  understates letters likely to occur in several places.
"""


import torch
import torch.nn as nn



class HangmanOutput(dict):
    """Model outputs, exposed as attributes for readability."""

    set_logits: torch.Tensor
    pos_logits: torch.Tensor | None
    count_logits: torch.Tensor | None
    value: torch.Tensor | None

    def __getattr__(self, name: str):  # pragma: no cover - trivial
        try:
            return self[name]
        except KeyError as exc:
            raise AttributeError(name) from exc


class HangmanFormer(nn.Module):
    """Character transformer with set / position / count / value heads."""

    def __init__(self, cfg: ModelConfig) -> None:
        super().__init__()
        self.cfg = cfg
        d = cfg.d_model

        self.tok_emb = nn.Embedding(VOCAB_SIZE, d)
        self.abs_pos = nn.Parameter(torch.zeros(1, SEQ_LEN, d)) if cfg.use_abs_pos else None
        if self.abs_pos is not None:
            nn.init.normal_(self.abs_pos, std=0.02)

        self.global_mlp = nn.Sequential(
            nn.Linear(GLOBAL_FEATURE_DIM, d),
            nn.SiLU(),
            nn.Linear(d, d),
        )
        self.film = (
            FiLMGenerator(GLOBAL_FEATURE_DIM, d, cfg.n_layers) if cfg.use_film else None
        )
        rope = (
            RotaryEmbedding(d // cfg.n_heads, SEQ_LEN) if cfg.use_rope else None
        )
        self.blocks = nn.ModuleList(
            [
                EncoderBlock(
                    d_model=d,
                    n_heads=cfg.n_heads,
                    d_ff=cfg.d_ff,
                    dropout=cfg.dropout,
                    # Linearly ramped stochastic depth, deeper blocks drop more.
                    drop_path=cfg.stochastic_depth * i / max(cfg.n_layers - 1, 1),
                    rope=rope,
                )
                for i in range(cfg.n_layers)
            ]
        )
        self.norm_out = RMSNorm(d)

        self.set_head = nn.Sequential(nn.Linear(d, d), nn.SiLU(), nn.Linear(d, 26))
        self.pos_head = nn.Linear(d, 26) if cfg.use_pos_head else None
        self.count_head = (
            nn.Linear(d, 26 * cfg.count_buckets) if cfg.use_count_head else None
        )
        self.value_head = (
            nn.Sequential(nn.Linear(d, d // 4), nn.SiLU(), nn.Linear(d // 4, 1))
            if cfg.use_value_head
            else None
        )
        self.apply(self._init_weights)

    @staticmethod
    def _init_weights(module: nn.Module) -> None:
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, std=0.02)

    # -- forward -------------------------------------------------------------

    def forward(
        self,
        tokens: torch.Tensor,
        attn_mask: torch.Tensor,
        globals: torch.Tensor,
        **_: torch.Tensor,
    ) -> HangmanOutput:
        x = self.tok_emb(tokens)
        if self.abs_pos is not None:
            x = x + self.abs_pos[:, : x.shape[1]]

        cond = self.global_mlp(globals)
        x = torch.cat([x[:, :1] + cond.unsqueeze(1), x[:, 1:]], dim=1)

        films = self.film(globals) if self.film is not None else [None] * len(self.blocks)
        for block, film in zip(self.blocks, films):
            x = block(x, attn_mask, film)
        x = self.norm_out(x)

        cls = x[:, 0]
        out = HangmanOutput(set_logits=self.set_head(cls))
        out["pos_logits"] = self.pos_head(x[:, 1:]) if self.pos_head is not None else None
        out["count_logits"] = (
            self.count_head(cls).view(-1, 26, self.cfg.count_buckets)
            if self.count_head is not None
            else None
        )
        out["value"] = (
            self.value_head(cls).squeeze(-1) if self.value_head is not None else None
        )
        return out

    # -- inference helpers ----------------------------------------------------

    noisy_or_presence = staticmethod(noisy_or_presence)

    @torch.no_grad()
    def presence_probs(
        self,
        batch: dict[str, torch.Tensor],
        hidden_mask: torch.Tensor | None = None,
        noisy_or_weight: float = 0.0,
    ) -> torch.Tensor:
        """``(B, 26)`` presence probability with guessed letters zeroed."""
        out = self(**batch)
        probs = torch.sigmoid(out.set_logits)
        if noisy_or_weight > 0.0 and out.pos_logits is not None and hidden_mask is not None:
            pooled = noisy_or_presence(out.pos_logits, hidden_mask)
            probs = (1.0 - noisy_or_weight) * probs + noisy_or_weight * pooled
        return probs.masked_fill(batch["guessed"], 0.0)

    # -- introspection ---------------------------------------------------------

    def parameter_report(self) -> list[tuple[str, int, str]]:
        """Layer-by-layer parameter table used in reports/MODEL_REPORT.md."""
        rows: list[tuple[str, int, str]] = []
        for name, child in self.named_children():
            n = count_parameters(child)
            if n:
                rows.append((name, n, child.__class__.__name__))
        rows.append(("TOTAL", count_parameters(self), ""))
        return rows

In [ ]:
# ====================================================================
# src/hangman/models/bigru.py
# ====================================================================
"""Bi-GRU ensemble member.

Architectural diversity contributes more to an ensemble than seed diversity
alone (Lakshminarayanan et al., 2017, *Simple and Scalable Predictive
Uncertainty Estimation using Deep Ensembles*).  A recurrent model makes
different mistakes from an attention model on the same board - in particular it
is stronger on short words, where there is little context for attention to
exploit and where the measured baseline loses most of its games.

The conditioning, heads and output contract are identical to
:class:`~hangman.models.hangmanformer.HangmanFormer`, so the two are
interchangeable everywhere downstream.
"""


import torch
import torch.nn as nn



class HangmanBiGRU(nn.Module):
    """Stacked bidirectional GRU with FiLM-style global conditioning."""

    def __init__(self, cfg: ModelConfig) -> None:
        super().__init__()
        self.cfg = cfg
        d = cfg.d_model
        half = d // 2

        self.tok_emb = nn.Embedding(VOCAB_SIZE, d)
        self.global_mlp = nn.Sequential(
            nn.Linear(GLOBAL_FEATURE_DIM, d), nn.SiLU(), nn.Linear(d, 2 * d)
        )
        nn.init.zeros_(self.global_mlp[-1].weight)
        nn.init.zeros_(self.global_mlp[-1].bias)

        self.rnn = nn.GRU(
            input_size=d,
            hidden_size=half,
            num_layers=cfg.rnn_layers,
            batch_first=True,
            bidirectional=True,
            dropout=cfg.dropout if cfg.rnn_layers > 1 else 0.0,
        )
        self.norm_out = RMSNorm(d)
        self.drop = nn.Dropout(cfg.dropout)

        self.set_head = nn.Sequential(nn.Linear(2 * d, d), nn.SiLU(), nn.Linear(d, 26))
        self.pos_head = nn.Linear(d, 26) if cfg.use_pos_head else None
        self.count_head = (
            nn.Linear(2 * d, 26 * cfg.count_buckets) if cfg.use_count_head else None
        )
        self.value_head = (
            nn.Sequential(nn.Linear(2 * d, d // 4), nn.SiLU(), nn.Linear(d // 4, 1))
            if cfg.use_value_head
            else None
        )

    def forward(
        self,
        tokens: torch.Tensor,
        attn_mask: torch.Tensor,
        globals: torch.Tensor,
        **_: torch.Tensor,
    ) -> HangmanOutput:
        x = self.tok_emb(tokens)
        gamma, beta = self.global_mlp(globals).chunk(2, dim=-1)
        x = x * (1.0 + gamma.unsqueeze(1)) + beta.unsqueeze(1)
        x = x * attn_mask.unsqueeze(-1).to(x.dtype)

        h, _ = self.rnn(x)
        h = self.norm_out(self.drop(h))

        valid = attn_mask.unsqueeze(-1).to(h.dtype)
        mean_pool = (h * valid).sum(dim=1) / valid.sum(dim=1).clamp_min(1.0)
        summary = torch.cat([h[:, 0], mean_pool], dim=-1)

        out = HangmanOutput(set_logits=self.set_head(summary))
        out["pos_logits"] = self.pos_head(h[:, 1:]) if self.pos_head is not None else None
        out["count_logits"] = (
            self.count_head(summary).view(-1, 26, self.cfg.count_buckets)
            if self.count_head is not None
            else None
        )
        out["value"] = (
            self.value_head(summary).squeeze(-1) if self.value_head is not None else None
        )
        return out

    noisy_or_presence = staticmethod(noisy_or_presence)

    def parameter_report(self) -> list[tuple[str, int, str]]:
        rows = [
            (name, count_parameters(child), child.__class__.__name__)
            for name, child in self.named_children()
            if count_parameters(child)
        ]
        rows.append(("TOTAL", count_parameters(self), ""))
        return rows

In [ ]:
# ====================================================================
# src/hangman/models/dilated_cnn.py
# ====================================================================
"""Dilated convolutional ensemble member.

A stack of dilated 1-D convolutions grows its receptive field exponentially
with depth (van den Oord et al., 2016, *WaveNet*; Kalchbrenner et al., 2016,
*Neural Machine Translation in Linear Time*).  On a 29-character board a depth
of eight already sees the whole word, while the inductive bias stays local -
which is exactly the bias that suits affixes and digraphs, and a genuinely
different bias from both attention and recurrence.

Conditioning, heads and output contract match the other two members.
"""


import torch
import torch.nn as nn



class _ResidualConvBlock(nn.Module):
    def __init__(self, d: int, kernel: int, dilation: int, dropout: float) -> None:
        super().__init__()
        pad = dilation * (kernel - 1) // 2
        self.norm = RMSNorm(d)
        self.conv = nn.Conv1d(d, 2 * d, kernel, padding=pad, dilation=dilation)
        self.proj = nn.Conv1d(d, d, 1)
        self.drop = nn.Dropout(dropout)
        self.film = nn.Linear(d, 2 * d)
        nn.init.zeros_(self.film.weight)
        nn.init.zeros_(self.film.bias)

    def forward(self, x: torch.Tensor, cond: torch.Tensor) -> torch.Tensor:
        h = self.norm(x.transpose(1, 2)).transpose(1, 2)
        gamma, beta = self.film(cond).chunk(2, dim=-1)
        h = h * (1.0 + gamma.unsqueeze(-1)) + beta.unsqueeze(-1)
        value, gate = self.conv(h).chunk(2, dim=1)
        return x + self.drop(self.proj(torch.tanh(value) * torch.sigmoid(gate)))


class HangmanDilatedCNN(nn.Module):
    """Gated dilated residual CNN over the character axis."""

    def __init__(self, cfg: ModelConfig) -> None:
        super().__init__()
        self.cfg = cfg
        d = cfg.d_model
        self.tok_emb = nn.Embedding(VOCAB_SIZE, d)
        self.global_mlp = nn.Sequential(
            nn.Linear(GLOBAL_FEATURE_DIM, d), nn.SiLU(), nn.Linear(d, d)
        )
        self.blocks = nn.ModuleList(
            [
                _ResidualConvBlock(d, cfg.cnn_kernel, 2 ** (i % 5), cfg.dropout)
                for i in range(cfg.cnn_layers)
            ]
        )
        self.norm_out = RMSNorm(d)
        self.set_head = nn.Sequential(nn.Linear(2 * d, d), nn.SiLU(), nn.Linear(d, 26))
        self.pos_head = nn.Linear(d, 26) if cfg.use_pos_head else None
        self.count_head = (
            nn.Linear(2 * d, 26 * cfg.count_buckets) if cfg.use_count_head else None
        )
        self.value_head = (
            nn.Sequential(nn.Linear(2 * d, d // 4), nn.SiLU(), nn.Linear(d // 4, 1))
            if cfg.use_value_head
            else None
        )

    def forward(
        self,
        tokens: torch.Tensor,
        attn_mask: torch.Tensor,
        globals: torch.Tensor,
        **_: torch.Tensor,
    ) -> HangmanOutput:
        cond = self.global_mlp(globals)
        x = self.tok_emb(tokens).transpose(1, 2)          # (B, d, S)
        mask = attn_mask.unsqueeze(1).to(x.dtype)
        x = x * mask
        for block in self.blocks:
            x = block(x, cond) * mask
        h = self.norm_out(x.transpose(1, 2))              # (B, S, d)

        valid = attn_mask.unsqueeze(-1).to(h.dtype)
        mean_pool = (h * valid).sum(dim=1) / valid.sum(dim=1).clamp_min(1.0)
        summary = torch.cat([h[:, 0], mean_pool], dim=-1)

        out = HangmanOutput(set_logits=self.set_head(summary))
        out["pos_logits"] = self.pos_head(h[:, 1:]) if self.pos_head is not None else None
        out["count_logits"] = (
            self.count_head(summary).view(-1, 26, self.cfg.count_buckets)
            if self.count_head is not None
            else None
        )
        out["value"] = (
            self.value_head(summary).squeeze(-1) if self.value_head is not None else None
        )
        return out

    noisy_or_presence = staticmethod(noisy_or_presence)

    def parameter_report(self) -> list[tuple[str, int, str]]:
        rows = [
            (name, count_parameters(child), child.__class__.__name__)
            for name, child in self.named_children()
            if count_parameters(child)
        ]
        rows.append(("TOTAL", count_parameters(self), ""))
        return rows

In [ ]:
ARCHITECTURES = {
    "transformer": HangmanFormer,
    "bigru": HangmanBiGRU,
    "dilated_cnn": HangmanDilatedCNN,
}

def build_model(cfg):
    return ARCHITECTURES[cfg.arch](cfg)

def load_checkpoint(path, device=DEVICE, use_ema=True):
    blob = torch.load(path, map_location=device, weights_only=False)
    cfg = ExperimentConfig.load_from_dict(blob["config"])
    model = build_model(cfg.model).to(device)
    model.load_state_dict(blob["model"])
    if use_ema and "ema" in blob:
        state = model.state_dict()
        for name, value in blob["ema"].items():
            if name in state:
                state[name].copy_(value.to(state[name].dtype))
    return model.eval(), cfg

## 3. Inference

Letter scores blend the presence head with a **noisy-OR pooling** of the per-position head,
`P(present) = 1 − Π_i(1 − p_i(ℓ))`. That pooling is not a detail: published implementations
max-pool here, and switching to noisy-OR was worth **+3.2 points** on the model (and +8.0 on the
n-gram baseline). The per-member blend weight was fitted on the held-out dictionary, never on
`test.txt`.

The two ensemble members are averaged with **equal weight**. The NLL-fitted calibration is *not*
applied — measured on these two members it puts 87.9% of the weight on the weaker one and costs
1.46 points. Fitting log-likelihood is not fitting win rate.

In [ ]:
class ModelSource:
    # Presence probabilities from one trained model.

    def __init__(self, model, noisy_or_weight=0.5):
        self.model = model
        self.noisy_or_weight = noisy_or_weight

    @torch.no_grad()
    def presence(self, board, lengths, present, absent):
        tokens, attn = encode_tokens(board)
        globals_vec = encode_globals(lengths, present, absent)
        with torch.autocast(board.device.type, dtype=AMP_DTYPE,
                            enabled=(board.device.type == "cuda")):
            out = self.model(tokens=tokens, attn_mask=attn, globals=globals_vec)
        probs = torch.sigmoid(out.set_logits.float())
        if self.noisy_or_weight > 0.0 and out.pos_logits is not None:
            hidden = board == HOLE
            seq = min(out.pos_logits.shape[1], hidden.shape[1])
            pooled = noisy_or_presence(out.pos_logits[:, :seq].float(), hidden[:, :seq])
            probs = (1.0 - self.noisy_or_weight) * probs + self.noisy_or_weight * pooled
        return probs.masked_fill(present | absent, 0.0)


class EnsembleSource:
    # Logit-space average of several models, with fitted weights and temperatures.

    def __init__(self, sources, weights=None, temperatures=None):
        self.sources = sources
        self.weights = weights
        self.temperatures = temperatures

    @torch.no_grad()
    def presence(self, board, lengths, present, absent):
        logits = []
        for i, src in enumerate(self.sources):
            p = src.presence(board, lengths, present, absent).clamp(1e-6, 1 - 1e-6)
            logit = torch.logit(p)
            if self.temperatures is not None:
                logit = logit / self.temperatures[i]
            logits.append(logit)
        stacked = torch.stack(logits, dim=0)
        if self.weights is not None:
            w = self.weights.view(-1, 1, 1)
            pooled = (stacked * w).sum(dim=0) / w.sum()
        else:
            pooled = stacked.mean(dim=0)
        return torch.sigmoid(pooled).masked_fill(present | absent, 0.0)


def greedy_policy(source):
    @torch.no_grad()
    def fn(board, lengths, present, absent, lives, active):
        return source.presence(board, lengths, present, absent).argmax(dim=1)
    return fn

## 4. Load the model

In [ ]:
checkpoints = sorted(WEIGHTS_DIR.glob("*_final.pt")) or sorted(WEIGHTS_DIR.glob("*.pt"))
if not checkpoints:
    raise SystemExit(
        f"no checkpoints under {WEIGHTS_DIR}. Attach the trained weights as a Kaggle dataset, "
        "or set TRAIN = True to train from train.txt."
    )

calib_path = WEIGHTS_DIR / "ensemble_calibration.pt"
calib = torch.load(calib_path, map_location=DEVICE, weights_only=False) if calib_path.exists() else {}
per_member_nor = calib.get("noisy_or_weights", {})

sources, names = [], []
for path in checkpoints:
    model, mcfg = load_checkpoint(path)
    weight = per_member_nor.get(path.stem, 0.5)
    sources.append(ModelSource(model, noisy_or_weight=weight))
    names.append(path.stem)
    n = sum(p.numel() for p in model.parameters())
    print(f"  {path.stem:<28} {mcfg.model.arch:<12} {n/1e6:6.2f}M  noisy_or_weight={weight}")

# --------------------------------------------------------------------------
# Members are averaged with EQUAL weight.  The fitted calibration is
# deliberately not applied: measured on these two members it puts 87.9% of the
# weight on the *weaker* model and scores 68.15% against 69.61% for a plain
# average.  It is fitted by minimising negative log-likelihood, which rewards
# being calibrated across all 26 letters, while the game only ever asks which
# letter ranks first.  See reports/MODEL_REPORT.md section 10a.
#
# The per-member noisy-OR blend *is* used - that one was fitted against win
# rate directly and is worth several points.
# --------------------------------------------------------------------------
USE_FITTED_CALIBRATION = False

if len(sources) == 1:
    source = sources[0]
    print("\nsingle model")
elif USE_FITTED_CALIBRATION and "weights" in calib and len(calib["weights"]) == len(sources):
    source = EnsembleSource(sources,
                            weights=calib["weights"].to(DEVICE),
                            temperatures=calib["temperatures"].to(DEVICE))
    print(f"\ncalibrated ensemble of {len(sources)}: weights={calib['weights'].tolist()}")
else:
    source = EnsembleSource(sources)
    print(f"\nuniform ensemble of {len(sources)} members (equal weight, as measured best)")

## 5. Simulate every test word

The competition asks for the chronological string of letters the policy *would* guess, so we play
each game locally under the exact rules and record what the policy actually did. Games advance in
lock-step batches grouped by length, so one forward pass serves thousands of games — the whole
250,000-word test set takes a few minutes.

Rules implemented exactly: all occurrences of a correct letter revealed at once; any character
that reveals nothing new costs one life (a repeated letter included); the **sixth** wrong guess is
a loss, five are survivable.

In [ ]:
from collections import defaultdict

test_words = [line.strip().lower() for line in open(TEST_FILE) if line.strip()]
print(f"{len(test_words):,} test words")

by_length = defaultdict(list)
for idx, word in enumerate(test_words):
    by_length[len(word)].append(idx)

policy = greedy_policy(source)
guesses = [None] * len(test_words)
n_won = total_wrong = 0
started = time.time()

for length in sorted(by_length):
    ids = by_length[length]
    for offset in range(0, len(ids), 8192):
        chunk_ids = ids[offset:offset + 8192]
        result = play_batch([test_words[i] for i in chunk_ids], policy, device=DEVICE)
        strings = result.guess_strings()
        for local, word_id in enumerate(chunk_ids):
            guesses[word_id] = strings[local]
        n_won += int(result.won.sum().item())
        total_wrong += result.total_wrong()

elapsed = time.time() - started
print(f"played {len(test_words):,} games in {elapsed:.0f}s")
print(f"self-scored win rate: {100 * n_won / len(test_words):.3f}%   E = {total_wrong:,}")

## 6. Write and independently re-validate the submission

In [ ]:
frame = pd.DataFrame({"word_id": range(len(test_words)), "guessed_letters_string": guesses})
frame.to_csv(OUT_CSV, index=False)
print(f"wrote {OUT_CSV}")

# Re-read the file and re-score it from scratch with a naive per-game scorer,
# rather than trusting the generator.  A disagreement here would mean the
# submission does not say what we think it says.
check = pd.read_csv(OUT_CSV, dtype={"word_id": int, "guessed_letters_string": str})
check["guessed_letters_string"] = check["guessed_letters_string"].fillna("")

assert list(check.columns) == ["word_id", "guessed_letters_string"]
assert len(check) == 250_000, f"expected 250,000 rows, found {len(check):,}"
assert check["word_id"].tolist() == list(range(len(check))), "word_id must be 0..249,999 in order"

alphabet = set("abcdefghijklmnopqrstuvwxyz")
won = wrong = 0
for word, guess_string in zip(test_words, check["guessed_letters_string"]):
    assert set(guess_string) <= alphabet, f"non-alphabet character in {guess_string!r}"
    assert len(set(guess_string)) == len(guess_string), f"repeated letter in {guess_string!r}"
    w, e, _ = reference_play(word, guess_string)
    won += int(w)
    wrong += e

win_rate = 100 * won / len(check)
print(f"validator: {won:,}/{len(check):,} solved = {win_rate:.3f}%   E = {wrong:,}")
print(f"leaderboard score (Score - E/1e8): {win_rate - wrong / 1e8:.6f}")
assert abs(win_rate - 100 * n_won / len(test_words)) < 1e-9, "generator and validator disagree"
print("\nall checks passed")
check.head(8)

## Notes for the reviewer

* **`test.txt` is used only to simulate games.** It never trains or tunes anything. Every
  hyperparameter — the teacher's backoff constant, the noisy-OR blend, the ensemble weights and
  temperatures, the inference policy — was selected on a 22,530-word slice held out of `train.txt`.
* **No lookup tables.** The model's only inputs at inference are the board, the two guessed-letter
  sets and the remaining lives.
* **Full training code** is in `hangman_end_to_end.ipynb` and `src/hangman/`; set `TRAIN = True`
  above, or run `bash scripts/03_train_all.sh`. Reports covering the architecture, the literature,
  the LLM-teacher stage and rule compliance are under `reports/`.